# 1. OpenPyTEA workflow and equipment costing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/01_equipment_costing.ipynb)

This notebook first shows how OpenPyTEA works: its workflow and architecture. We then use the `Equipment` class to turn equipment sizes into **purchased** and **installed (direct) costs**, and build the equipment list of the hypothetical green-ammonia synthesis plant used for the rest of the session.

## Learning goals

- Understand the OpenPyTEA workflow and which module does what.
- Cost equipment with `Equipment`: correlations, inflation, and installation factors.
- Find the right category, type, and size units in the cost-correlation database.
- Use vendor quotes, change cost year, and handle oversized equipment with parallel units.
- Build the equipment list of the demo plant.

## 0) Setup

In [ ]:
# Run this cell. It installs OpenPyTEA if it is missing (e.g., on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

## 1) How OpenPyTEA works

![OpenPyTEA architecture](https://raw.githubusercontent.com/pbtamarona/OpenPyTEA/main/examples/img/architecture.png)

OpenPyTEA links process-simulation and equipment-sizing results to a plant-level economic evaluation. As the figure shows, you supply three kinds of input:

- **Economic assumptions**: feed and energy prices, project lifetime, interest and tax rates, location, and so on.
- **Process simulation results** (Aspen Plus, CHEMCAD, or a spreadsheet): the mass and energy balance gives feed and utility consumption, product rates, and emission streams.
- **Equipment sizing**: compressor power, reactor volume, heat-exchanger area, and so on.

OpenPyTEA turns them into costs and economic metrics in three steps. Each step is a module of the package and a notebook of this workshop:

| Step | Module and main objects | What it does | Notebook |
|----------------|------------------------------|-------------------------------------------|-----------|
| 1. Equipment | `equipment.py`: `Equipment`, `CompositeEquipment` | Size → purchased cost → inflation-adjusted → direct cost | 1 |
| 2. Plant | `plant.py`: `Plant` | Combines equipment and assumptions into CAPEX, OPEX, cash flows, and metrics (NPV, IRR, LCOP, ...) | 2 |
| 3. Analysis | `analysis.py` + `plotting.py` | Cost breakdowns, sensitivity analysis, and Monte Carlo uncertainty on one or several `Plant` objects | 3, 4, 5 |
| Input/output | `io.py`, `cli.py` | JSON configuration files, exporting results, command line | 6 |

Two plain CSV files shipped with the package hold the data: `cost_correlations.csv` (equipment cost correlations with size units, validity ranges, cost years, and sources) and `cepci_values.csv` (annual averages of the Chemical Engineering Plant Cost Index, CEPCI). Because they are plain files, you can review every number and add your own.

Equipment and plant are kept separate on purpose: one `Equipment` object can be reused in several `Plant`
configurations (for example, two scenarios with different prices), and the analysis functions accept a list of plants so different technologies can be compared side by side (Notebook 5). In this notebook we zoom in on step 1.

## 2) From size to cost in three steps

Step 1 of the workflow (`equipment.py`) does the following for each equipment item:

1. **Purchased cost** $C_p$ from a cost correlation selected by `category` and `type`. The database (`cost_correlations.csv`) stores 400+ correlations from Towler & Sinnott, Turton et al., Seider et al., Ulrich, Perry, and journal papers, each with its size parameter, validity range, and cost year.
2. **Inflation adjustment** to the target year (2024 by default) with CEPCI:
   \begin{equation}
   C_{\text{target}} = C_{\text{base}} \cdot \frac{\text{CEPCI}_{\text{target}}}{\text{CEPCI}_{\text{base}}}
   \end{equation}
3. **Direct (installed) cost** with the factorial method of Towler & Sinnott:
   \begin{equation}
   C_D = C_p \left[(1 + f_p) f_m + (f_{er} + f_{el} + f_i + f_c + f_s + f_l)\right]
   \end{equation}
   where $f_m$ is the material factor and the other factors (piping, erection, electrical, instrumentation, civil, structural, lagging) depend on the `process_type` (`Fluids`, `Solids`, `Mixed`, `Electrical`).

Let us cost a shell-and-tube heat exchanger of 250 m² in 304 stainless steel:

In [ ]:
from openpytea import Equipment

exchanger = Equipment(
    name="E-101",
    param=250,  # heat-transfer area in m2 (the units come from the correlation)
    process_type="Fluids",
    category="Heat exchangers",
    type="U-tube shell & tube",
    material="304 stainless steel",
)

print(exchanger)
print(f"Purchased cost (2024 USD): {exchanger.purchased_cost:,.0f}")
print(f"Direct cost    (2024 USD): {exchanger.direct_cost:,.0f}")
print(f"Material factor          : {exchanger.material_factor}")

The direct cost is about 3.7 times the purchased cost: installing the equipment (piping, erection, electrical,
instrumentation, and so on) costs more than the equipment itself. The installation factors used are class
attributes, so you can always inspect (or override) them:

In [ ]:
import pandas as pd

print("Installation factors per process type:")
display(pd.DataFrame(Equipment.process_factors))

print("A few material factors:")
print({m: Equipment.material_factors[m]
       for m in ["Carbon steel", "304 stainless steel", "316 stainless steel",
                 "Hastelloy C", "Nickel"]})

## 3) Finding the right correlation

A common question is *"what category, type, and units do I use?"*. The answer is in the cost-correlation database: open `cost_correlations.csv` in a spreadsheet, or browse it in the [OpenPyTEA documentation](https://openpytea.readthedocs.io/en/latest/user_guide/cost_correlations.html).

Here we load and filter it with pandas. The `units` column tells you what `param` must be.

In [ ]:
from importlib.resources import files

correlations = pd.read_csv(
    files("openpytea") / "data" / "cost_correlations.csv")
print(f"{len(correlations)} correlations in "
      f"{correlations['category'].nunique()} categories")

# All categories
print(sorted(correlations["category"].unique()))

In [ ]:
# The first 20 correlations of the compressors category:
# type, size units, validity range, cost year, and source
columns = ["type", "units", "s_lower", "s_upper", "cost_year", "source"]
mask = correlations["category"].str.contains("Compressors", case=False)
correlations[mask][columns].head(20)

> **Note.** When several correlations share the same category and type, OpenPyTEA takes the first one in the file. Pass `cost_func="<key>"` to pick a specific one.

## 4) Materials, vendor quotes, cost years, and assemblies

**4a. Material and process type change the installed cost, not the purchased cost**

In [ ]:
for material in ["Carbon steel", "304 stainless steel", "Hastelloy C"]:
    e = Equipment(name="E-101", param=250, process_type="Fluids",
                  category="Heat exchangers", type="U-tube shell & tube",
                  material=material)

    print(f"{material:<20} purchased {e.purchased_cost:>10,.0f}   "
          f"direct {e.direct_cost:>10,.0f}")

**4b. A vendor quote or literature value replaces the correlation** (give `purchased_cost` and its `cost_year`, OpenPyTEA escalates it to the target year)

In [ ]:
electrolyzer_stack = Equipment(
    name="Electrolyzer stack",
    param=None,  # not needed when the cost is given directly
    process_type="Fluids",
    category="Electrolysis",  # free label: this category is not in the database
    type="Alkaline stack",
    purchased_cost=17_194_000,  # quote in 2020 USD (DOE report)
    cost_year=2020,
)
print(f"2020 quote escalated to 2024: "
      f"{electrolyzer_stack.purchased_cost:,.0f} USD")

**4c. Costs in another year** (`target_year`) and **oversized equipment** (automatic parallel units when `param` exceeds the correlation's upper bound)

In [ ]:
exchanger_2019 = Equipment(name="E-101", param=250, process_type="Fluids",
                           category="Heat exchangers",
                           type="U-tube shell & tube",
                           material="304 stainless steel", target_year=2019)
print(f"Same exchanger in 2019 USD: {exchanger_2019.purchased_cost:,.0f} "
      f"(2024 USD: {exchanger.purchased_cost:,.0f})")

big_compressor = Equipment(name="C-201", param=45_000, process_type="Fluids",
                           category="Compressors, fans, & blowers",
                           type="Compressor, centrifugal")
print(f"45 MW compressor -> {big_compressor.num_units} parallel units, "
      f"direct cost {big_compressor.direct_cost:,.0f} USD")

**4d. Assemblies** such as a pressure-swing adsorption (PSA) skid can be built from sub-components with
`CompositeEquipment`. The composite behaves like a single equipment item in the plant.

In [ ]:
from openpytea import CompositeEquipment

# two identical vessels of 8 m3
vessels = Equipment(name="Adsorber vessels", param=8.0, process_type="Fluids",
                    category="Pressure vessels", type="Vertical",
                    material="Carbon steel", num_units=2)
# 2 x 150 ft3 of adsorbent
zeolite = Equipment(name="Zeolite 5A", param=2 * 150.0, process_type="Solids",
                    category="Packings & adsorbents", type="Molecular sieves")
psa = CompositeEquipment(name="PSA unit", process_type="Fluids",
                         components=[vessels, zeolite],
                         category="Adsorption", type="Pressure-swing adsorber")
print(psa)
psa.breakdown().round(0)

## 5) Task 1.1: cost the green-ammonia synthesis loop

The demo plant of this workshop is a small **green-ammonia synthesis loop** (100 t NH₃/day) fed with hydrogen
from an electrolyzer and nitrogen from an air separation unit. 

![Block diagram of the green-ammonia synthesis loop](assets/ammonia_loop.png)

The process simulation gives the following equipment sizes:

| Tag | Equipment | Category | Type | Size (`param`) | Material |
|-------|------------|----------------|-----------------|-----------|----------|
| C-101 | Syngas compressor | `Compressors, fans, & blowers` | `Compressor, centrifugal` | 1 200 kW driver power | Carbon steel |
| M-101 | Compressor motor | `Motors & generators` | `Totally enclosed motor` | 1 200 / 0.90 kW shaft power | default |
| E-101 | Feed/effluent exchanger | `Heat exchangers` | `U-tube shell & tube` | 250 m² | 304 stainless steel |
| R-101 | Ammonia converter | `Reactors` | `Tubular fixed bed` | 12 000 kg shell mass | 304 stainless steel |
| V-101 | Ammonia separator | `Pressure vessels` | `Vertical` | 25 m³ | Carbon steel |
| K-101 | Refrigeration unit | `Utilities` | `Packaged mechanical refrigerator` | 800 kW evaporator duty | default |
| P-101 | Product pump | `Pumps` | `Single-stage centrifugal` | 20 L/s | Carbon steel |

The motor is an electrical item, so use `process_type="Electrical"` for it; everything else is `"Fluids"`.

**Tasks**

1. Create the seven `Equipment` objects and put them in a list called `equipment_list`.
2. Print the name, purchased cost, and direct cost of each item, and the total direct cost.
3. Which item dominates? Does that match your engineering intuition?

In [ ]:
# YOUR CODE HERE
# Hints:
#   Equipment(name=..., param=..., process_type=..., category=..., type=...,
#   material=...)
#   Leave `material` out for the motor and the refrigeration unit
#   (the correlation's default material is used).
#   pd.DataFrame([{"name": e.name, "purchased": e.purchased_cost, "direct":
#   e.direct_cost}
#                 for e in equipment_list])

## 6) Task 1.2 (bonus): how sensitive is the estimate to your choices?

Using the ammonia converter (R-101), answer the following:

1. What is the direct cost in carbon steel instead of 304 stainless steel? By what factor does it change?
2. What is the purchased cost expressed in 2015 USD instead of 2024 USD?
3. Look up the validity range of the `Tubular fixed bed` correlation in the database. Is 12 000 kg inside it?

In [ ]:
# YOUR CODE HERE

## Recap

- `Equipment` turns a size into a purchased cost (correlation + CEPCI) and a direct cost (installation factors).
- Check the size units and validity range in the database before setting `param`.
- Vendor quotes go in via `purchased_cost` and `cost_year`; assemblies via `CompositeEquipment`.

**Next:** [2. Building a plant](02_building_a_plant.ipynb)